In [1]:
%run /home/hadoop/agrim_cdp/common_utils/db.ipynb

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

In [3]:
# --- Config ---
bucket = "agrim-cdp"
prefix = "data-landing/trackier/"
batch_size = 1000

In [4]:
# Unique identifier columns
EVENT_KEY_COLUMN = "event_name"
S3_REGION = 'ap-south-1'
table_secondary = "cdp_raw_db.trackier_install_events"
table_primary = "cdp_raw_db.trackier_inapp_events"

In [5]:
import os
os.environ["SPARK_HOME"] = "/home/hadoop/.local/lib/python3.9/site-packages/pyspark"  # Or wherever your Spark is
os.environ["PATH"] = os.environ["SPARK_HOME"] + "/bin:" + os.environ["PATH"]

In [6]:
spark = SparkSession.builder \
    .appName("webengage_batch_load") \
    .config("spark.jars", "/home/hadoop/agrim_cdp/common_jars/postgresql-42.2.24.jar") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
s3 = boto3.client('s3', region_name=S3_REGION)

:: loading settings :: url = jar:file:/home/hadoop/.local/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hadoop/.ivy2/cache
The jars for the packages stored in: /home/hadoop/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b04fad71-87e1-4cee-8445-8d581a3eae8d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in spark-list
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in spark-list
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in spark-list
:: resolution report :: resolve 177ms :: artifacts dl 6ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from spark-list in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from spark-list in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from spark-list in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| s

In [7]:
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=batch_size)
file_paths = []

In [8]:
for obj in response.get("Contents", []):
    key = obj["Key"]
    if key.endswith(".txt"):
        file_paths.append(f"s3a://{bucket}/{key}")

print(f"Processing {len(file_paths)} files...")

Processing 999 files...


In [9]:
RDS_HOST, RDS_DBNM, RDS_USER, RDS_PASSWORD, RDS_PORT = get_rds_credentials()

In [10]:
event_columns = [
    "ad", "ad_id", "ad_set", "ad_set_id", "camp", "camp_id", "channel", "cost", "cost_currency",
    "gaid", "geo", "install_time_unix", "os_version", "p1", "p2", "p3", "p4", "p5",
    "partner_id", "partner_name", "agency_id", "agency_name", "platform", "site_id", "sub_site_id",
    "trackier_install_id", "trackier_click_id", "trackier_impression_id",
    "sdk_param_1", "sdk_param_2", "sdk_param_3", "sdk_param_4", "sdk_param_5",
    "sdk_param_6", "sdk_param_7", "sdk_param_8", "sdk_param_9", "sdk_param_10",
    "source_ip", "event_id", "event_name", "event_revenue", "event_revenue_currency",
    "event_time_unix", "event_value", "is_reengagement", "order_id", "discount", "trackier_iae_id",
    "create_ts"
]

In [11]:
install_columns = [
    "ad", "ad_id", "ad_set", "ad_set_id", "camp", "camp_id", "channel", "cost", "cost_currency",
    "gaid", "geo", "install_time_unix", "os_version", "p1", "p2", "p3", "p4", "p5",
    "partner_id", "partner_name", "agency_id", "agency_name", "platform", "site_id", "sub_site_id",
    "trackier_install_id", "trackier_click_id", "trackier_impression_id",
    "sdk_param_1", "sdk_param_2", "sdk_param_3", "sdk_param_4", "sdk_param_5",
    "sdk_param_6", "sdk_param_7", "sdk_param_8", "sdk_param_9", "sdk_param_10",
    "source_ip", "create_ts"
]

In [12]:
# --- Process each file individually ---
for file_path in file_paths:
    TABLE_NAME=''
    try:
        df = spark.read.json(file_path)
        columns = df.columns
        df = df.withColumn("create_ts", current_timestamp())

        if EVENT_KEY_COLUMN in columns:
            TABLE_NAME = table_primary
            COLUMNS = event_columns
        else:
            TABLE_NAME = table_secondary
            COLUMNS = install_columns
        if not TABLE_NAME == '':
            df = df.select(COLUMNS)
            # ---- Step 4: Write to RDS using JDBC ----
            df.write \
                .format("jdbc") \
                .option("url", f'jdbc:postgresql://{RDS_HOST}:{RDS_PORT}/{RDS_DBNM}') \
                .option("dbtable", TABLE_NAME) \
                .option("user", RDS_USER) \
                .option("password", RDS_PASSWORD) \
                .option("driver", "org.postgresql.Driver") \
                .mode("append") \
                .save()

            key = "/".join(file_path.split("/")[3:])
            key_arch = key.replace("data-landing", "data-archive")
            
            s3.copy_object(
            Bucket=bucket,
            CopySource={'Bucket': bucket, 'Key': key},
            Key=key_arch
            )
            
            s3.delete_object(Bucket=bucket, Key=key)

    except Exception as e:
        print(f"❌ Error processing {file_path}: {str(e)}")

print("✅ Ingestion for this run complete.")

25/07/17 15:37:50 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/07/17 15:37:51 WARN CredentialsLegacyConfigLocationProvider: Found the legacy config profiles file at [/home/hadoop/.aws/config]. Please move it to the latest default location [~/.aws/credentials].
25/07/17 15:37:56 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

✅ Ingestion for this run complete.
